# Project 02 — BROKEN notebook (debugging exercise)

This notebook contains **seeded bugs** centred on the project's pitfall: scale priors. Run it, read the diagnostics, find each bug, and fix it. The clean reference is `notebook.ipynb`; the answer key is `BROKEN_BUGS.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
RNG = 20240602

In [ ]:
from data.generate_data import generate
data = generate()
y = data['y']

### Model — an improper flat scale prior, and a variance/SD confusion.

In [ ]:
# BUG 1: improper 'flat' prior on sigma via a huge Uniform(0, 1e6).
#         This is NOT uninformative; it puts vast mass on absurd spreads.
# BUG 2: passing a VARIANCE where pm.Normal expects a standard deviation.
with pm.Model() as model:
    mu = pm.Normal('mu', mu=5.0, sigma=10.0)
    sigma = pm.Uniform('sigma', lower=0.0, upper=1e6)   # BUG 1
    variance = sigma ** 2
    pm.Normal('y', mu=mu, sigma=variance, observed=y)   # BUG 2: variance, not sd
    idata = pm.sample(draws=800, tune=800, chains=2, random_seed=RNG,
                      progressbar=False)

In [ ]:
print(az.summary(idata, var_names=['mu', 'sigma']))
print('divergences:', int(idata.sample_stats['diverging'].sum()))

### Posterior predictive — BUG 3: reducing over the wrong axis for the spread.

In [ ]:
with model:
    idata.extend(pm.sample_posterior_predictive(idata, random_seed=RNG,
                                                progressbar=False))
pp = idata.posterior_predictive['y']
# BUG 3: std over 'draw' collapses across posterior samples, not observations
pp_sd = pp.std(dim='draw').values.ravel()
print('observed sd =', y.std(ddof=1), 'predicted sd mean =', pp_sd.mean())